[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/05_humanness/05_humanness_lab.ipynb)


# 05 — humanness & humanization (Sapiens 직접 실행)

> 본문: [`05_humanness.md`](05_humanness.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 16초** (실측)

**이 노트북은 도구를 직접 돌립니다.** Sapiens 언어모델을 **직접 돌려** humanness 점수·humanized 서열을 `my_run/` 에 만들어요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "05_humanness"
PIP_PKGS = "pandas matplotlib sapiens abnumber anarci"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

if shutil.which("hmmscan") is None:      # ANARCI 가 부르는 실행파일 — pip 로는 안 깔려요
    if IN_COLAB:
        _APT.append("hmmer")
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")           # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨집니다

if _APT:
    _run("apt-get -qq update", quiet=True)   # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), quiet=True)

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — Sapiens humanness + humanization (본문 5.1)

BioPhi CLI(`biophi sapiens`)는 **bioconda 전용**이라 Colab 에서 못 써요. 하지만 BioPhi 가 내부에서 쓰는
두 부품(`sapiens` 언어모델, `abnumber` numbering)은 **둘 다 pip 에 있어요** — 그래서 같은 알고리즘을
그대로 돌릴 수 있습니다.

```bash
python scripts/sapiens_humanize.py data/demo_mab.fa \
    --scores-out my_run/demo_sapiens_scores.csv --fasta-out my_run/demo_humanized.fa
```
① 위치별 아미노산 확률 예측 → ② 각 위치 최대확률 아미노산으로 재구성 → ③ **원본 CDR 재이식**
(BioPhi 기본값 — scheme=kabat, cdr_definition=kabat, 1 iteration, CDR 은 humanize 하지 않음)

In [ ]:
run_tool([PY, SCRIPTS/"sapiens_humanize.py", "data/demo_mab.fa",
          "--scores-out", "my_run/demo_sapiens_scores.csv",
          "--fasta-out",  "my_run/demo_humanized.fa"])

## 2) 내 결과 확인 — 사슬별 humanness (본문 5.2)

humanness 는 **Sapiens 가 입력 잔기에 준 확률**을 사슬 평균한 값이에요. 본문 5.2 그래프의 빨간 점선과
같은 **0.8** 을 기준선으로 씁니다.

In [ ]:
import pandas as pd

AA20 = list("ACDEFGHIKLMNPQRSTVWY")
HUMAN_LIKE = 0.8        # 본문 5.2 그래프의 빨간 기준선

def chain_humanness(df):
    """사슬별 mean P(input residue). X·gap 처럼 20종 밖 잔기는 평균에서 뺀다."""
    cols = [a for a in AA20 if a in df.columns]          # 컬럼 교집합 — 스키마가 달라도 안 죽게
    p = [row[row["input_aa"]] if row["input_aa"] in cols else None
         for _, row in df.iterrows()]
    return df.assign(p_input=p).groupby("chain")["p_input"].mean()

scores_csv = resolve("demo_sapiens_scores.csv")
scores = pd.read_csv(scores_csv)
hum = chain_humanness(scores)

for chain, v in hum.items():
    n = int((scores["chain"] == chain).sum())
    print(f"chain {chain}: humanness {v:.4f}  ({n} 잔기)  기준선 {HUMAN_LIKE} "
          f"{'위' if v >= HUMAN_LIKE else '아래'}")

low = [chain for chain, v in hum.items() if v < HUMAN_LIKE]
print(f"\n기준선 아래 사슬 = {low or '없음'} → humanize 로 손볼 대상이에요.")
print("Ch.04 에서 ANARCI 가 germline 을 판정한 사슬과 같은 쪽이 걸리는지 맞춰 보세요 "
      "(germline 할당 + 언어모델, 서로 독립인 두 도구의 합의).")

## 3) 내 결과 확인 — humanness 그래프 (본문 5.2)

왼쪽 패널이 방금 찍은 사슬별 평균 humanness(빨간 점선 = 0.8), 오른쪽 패널이 원본 → humanized 변이 수예요.
두 패널을 나란히 두면 "humanness 가 낮은 사슬을 더 많이 손본다"가 눈으로 확인됩니다.

In [ ]:
from antibody_viz import humanness_overview
from IPython.display import Image, display

png = "my_run/05_humanness_overview.png"
humanized_fa = resolve("demo_humanized.fa")
humanness_overview(scores_csv, "data/demo_mab.fa", humanized_fa,
                   "Humanness — Sapiens (demo mAb, 내 실행 결과)", png)
display(Image(png))

verdict = " · ".join(f"{chain} {v:.3f} {'≥' if v >= HUMAN_LIKE else '<'} {HUMAN_LIKE}"
                     for chain, v in hum.items())
print("기준선 대비 판정 —", verdict)
print(f"→ 기준선 {HUMAN_LIKE} 아래 {len([v for v in hum if v < HUMAN_LIKE])}개 사슬 / "
      f"위 {len([v for v in hum if v >= HUMAN_LIKE])}개 사슬.")

## 4) 어느 위치가 바뀌었나 — Kabat 번호와 FR/CDR 귀속 (본문 5.3 · 5.4)

본문 5.3 은 변이 **수**를, 5.4 는 **위치**를 묻습니다 — Vernier zone·CDR-supporting residue 를 건드리면
humanness 가 올라가도 결합을 잃을 수 있으니까요. `abnumber` 로 Kabat 번호를 붙여 두 질문을 한 표로 닫아요.

In [ ]:
import pandas as pd

def read_fa(path):
    d, name = {}, None
    for line in open(path):
        line = line.strip()
        if line.startswith(">"):
            name = line[1:].split()[0]; d[name] = ""
        elif name:
            d[name] += line
    return d

orig = read_fa("data/demo_mab.fa")
humanized = read_fa(humanized_fa)          # 3절이 고른 그 FASTA

try:
    from abnumber import Chain
except Exception as e:                       # abnumber 는 hmmscan 이 있어야 돌아요
    Chain = None
    print(f"[abnumber 사용 불가 — {type(e).__name__}] Kabat 번호 없이 raw 위치만 표시합니다.")
    print("→ pip install abnumber anarci 와 hmmscan(conda install -c bioconda hmmer) 후 재실행하세요.")

rows = []
for sid, o in orig.items():
    h = humanized.get(sid, "")
    if len(o) != len(h):
        print(f"{sid}: 길이가 달라(원본 {len(o)} / humanized {len(h)}) 위치 비교를 건너뜁니다.")
        continue
    labels = {}
    if Chain is not None:
        try:
            ab = Chain(o, scheme="kabat", cdr_definition="kabat")
            ct = "H" if ab.is_heavy_chain() else "L"
            off = o.find(ab.seq)             # abnumber 가 V 도메인 밖을 잘라내면 생기는 offset
            for i, pos in enumerate(ab.positions):
                labels[off + i] = (format(pos), pos.get_region().replace("CDR", f"CDR-{ct}"))
        except Exception as e:
            print(f"{sid}: numbering 실패({type(e).__name__}) → raw 위치만 표시합니다.")
    for i, (a, b) in enumerate(zip(o, h)):
        if a != b:
            num, region = labels.get(i, ("-", "-"))
            rows.append({"seq": sid, "raw_pos": i + 1, "Kabat": num,
                         "region": region, "original": a, "humanized": b})

diff = pd.DataFrame(rows, columns=["seq", "raw_pos", "Kabat", "region", "original", "humanized"])
for sid, o in orig.items():
    n = int((diff["seq"] == sid).sum()) if len(diff) else 0
    print(f"{sid}: 길이 {len(o)} · 변이 {n}개 ({n / len(o) * 100:.0f}%)")

if len(diff):
    display(diff)
    n_cdr = int(diff["region"].astype(str).str.startswith("CDR").sum())
    vc = diff["region"].value_counts()
    print("region 분포 —", " · ".join(f"{k} {v}개" for k, v in sorted(vc.items())))
    if n_cdr == 0:
        print(f"→ 변이 {len(diff)}개가 전부 framework 예요. 1절 ③ '원본 CDR 재이식'이 그대로 작동했습니다 "
              f"— 결합부위는 손대지 않았어요.")
    else:
        print(f"→ CDR 안 변이가 {n_cdr}개 있어요. 결합을 잃었을 수 있으니 본문 5.4 workflow 의 "
              f"6단계(back-mutation 후보)로 올리세요.")
else:
    print("\n바뀐 위치가 없어요 — Sapiens 가 '고칠 필요 없다'고 본 서열이에요.")

## 5) 레퍼런스 대조 — BioPhi CLI 결과와 같은가 (본문 5.1)

`data/demo_sapiens_scores.csv`·`data/demo_humanized.fa` 는 **bioconda BioPhi CLI** 로 만들어 커밋한 대조군이에요.
여기서 보는 판정 기준은 딱 두 개 — **사슬 평균 humanness 를 소수 4자리까지** 비교한 차이,
그리고 **humanized 서열이 글자 그대로 같은지** 입니다.

In [ ]:
mine_csv, mine_fa = MYRUN/"demo_sapiens_scores.csv", MYRUN/"demo_humanized.fa"

if mine_csv.exists() and mine_fa.exists():
    mine, ref = pd.read_csv(mine_csv), pd.read_csv("data/demo_sapiens_scores.csv")
    cmp = pd.DataFrame({"내 결과": chain_humanness(mine).round(4),
                        "레퍼런스(BioPhi CLI)": chain_humanness(ref).round(4)})
    cmp["차이"] = (cmp["내 결과"] - cmp["레퍼런스(BioPhi CLI)"]).abs().round(4)
    display(cmp)

    m_fa, r_fa = read_fa(mine_fa), read_fa("data/demo_humanized.fa")
    for sid in r_fa:
        print(f"{sid}: 서열 동일 = {m_fa.get(sid) == r_fa.get(sid)}")

    max_gap = float(cmp["차이"].max())
    same_seq = all(m_fa.get(sid) == r_fa.get(sid) for sid in r_fa)
    print(f"\n판정 — 사슬 평균 차이 최대 {max_gap:.4f} (소수 4자리) · humanized 서열 동일 = {same_seq}")
    if max_gap == 0.0 and same_seq:
        print("→ pip 부품 조합(sapiens + abnumber)이 bioconda CLI 를 재현했어요.")
    else:
        print("→ 차이가 남았어요. sapiens·abnumber 버전과 scheme(kabat) 설정부터 확인하세요.")
else:
    print("my_run 산출물이 없어 대조를 건너뜁니다 (1절이 실패했거나 건너뛰었어요).")

> 다음 → 본문 [06. 구조예측 (IgFold)](../06_structure/06_structure.md)